# DB Sanity Check
Quick debug cells to verify ingestion output is clean and readable.

In [6]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

conn = sqlite3.connect('../scores.db')

perf_count = conn.execute('SELECT COUNT(*) FROM performances').fetchone()[0]
score_count = conn.execute('SELECT COUNT(*) FROM scores').fetchone()[0]
print(f'performances: {perf_count}')
print(f'scores:       {score_count}')

performances: 4441
scores:       56910


## All Ensembles

In [7]:
ensembles = pd.read_sql("""
    SELECT ensemble_name, ensemble_slug, ensemble_location,
           COUNT(*) AS appearances,
           GROUP_CONCAT(DISTINCT class_code) AS classes
    FROM performances
    GROUP BY ensemble_slug
    ORDER BY ensemble_name
""", conn)

print(f'{len(ensembles)} unique ensembles')
ensembles

259 unique ensembles


,ensemble_name,ensemble_slug,ensemble_location,appearances,classes
0,AB Miller HS,ab_miller_hs,NaN,1,psca
1,Aliso Niguel HS,aliso_niguel_hs,"Aliso Viejo, CA",41,"psa,psca"
2,Alta Loma HS,alta_loma_hs,"Rancho Cucamonga, CA",3,psa
3,Alvarado IS,alvarado_is,NaN,14,psj
4,Arcadia HS,arcadia_hs,"Arcadia, CA",99,"psa,psw,pso"
...,...,...,...,...,...
254,Woodbridge HS,woodbridge_hs,"Irvine, CA",49,"pso,pscw,psa"
255,Woodcrest JHS,woodcrest_jhs,NaN,11,psj
256,Workman HS,workman_hs,NaN,11,"psco,psca"
257,Yorba Linda HS,yorba_linda_hs,NaN,13,psca


## Unique Performance Days

In [8]:
days = pd.read_sql("""
    SELECT performance_date,
           GROUP_CONCAT(DISTINCT competition_name) AS competitions,
           COUNT(*) AS performances
    FROM performances
    GROUP BY performance_date
    ORDER BY performance_date
""", conn)

days

,performance_date,competitions,performances
0,2017-02-04,Valencia HS,25
1,2017-02-11,Temescal Canyon HS,34
2,2017-02-25,Damien HS,34
3,2017-03-04,Monrovia HS,35
4,2017-03-05,Monrovia HS,33
...,...,...,...
112,2026-03-22,Eleanor Roosevelt HS,34
113,2026-03-28,Championship Prelims,59
114,2026-03-29,Championship Prelims,49
115,2026-04-04,"Championship Prelims, Semifinals, & PSJ Finals",47


## Full Block: PSW Championship Prelims (2026-03-29)

Percussion Scholastic World class at Championship Prelims, sorted by total score.

In [9]:
block = pd.read_sql("""
    SELECT
        total_rank        AS rank,
        placement         AS place,
        ensemble_name,
        ensemble_location,
        subtotal_score,
        penalty_score,
        total_score
    FROM performances
    WHERE class_code = 'psw'
      AND competition_name = 'Championship Prelims'
      AND performance_date = '2026-03-29'
    ORDER BY total_score DESC
""", conn)

block

,rank,place,ensemble_name,ensemble_location,subtotal_score,penalty_score,total_score
0,1,1,Arcadia HS,"Arcadia, CA",93.550,0.0,93.550
1,2,2,Chino Hills HS,"Chino Hills, CA",93.400,0.0,93.400
2,3,3,Vista Murrieta HS,"Murrieta, CA",92.775,0.0,92.775
3,4,4,Ayala HS,"Chino Hills, CA",92.300,0.0,92.300
4,5,5,Etiwanda HS,"Rancho Cucamonga, CA",87.750,0.0,87.750
5,6,6,Clovis East HS,"Clovis, CA",84.250,0.0,84.250
6,7,7,El Dorado HS,"Placentia, CA",83.950,0.0,83.950
7,8,8,Rowland HS,"Rowland Heights, CA",83.700,0.0,83.700


## Caption scores for one ensemble (spot check)

Drill into a single performance to verify the scores table is populated correctly.

In [10]:
spot = pd.read_sql("""
    SELECT s.caption, s.subcaption, s.role, s.score, s.rank, s.judge, s.judge_slot
    FROM scores s
    JOIN performances p ON s.performance_key = p.performance_key
    WHERE p.ensemble_name = 'Arcadia HS'
      AND p.performance_date = '2026-03-29'
      AND p.class_code = 'psw'
    ORDER BY s.caption, s.role, s.judge_slot
""", conn)

spot

,caption,subcaption,role,score,rank,judge,judge_slot
0,effect_music,total,judge_total,28.05,1,M. McIntosh,1
1,effect_music,overall,raw_score,95.00,1,M. McIntosh,1
2,effect_music,music,raw_score,92.00,3,M. McIntosh,1
3,effect_visual,total,judge_total,18.90,2,T. Fairbanks,1
4,effect_visual,overall,raw_score,94.00,2,T. Fairbanks,1
5,effect_visual,visual,raw_score,95.00,1,T. Fairbanks,1
6,music,total,judge_total,28.30,2,T. Rarick,1
7,music,composition,raw_score,95.00,2,T. Rarick,1
8,music,performance,raw_score,94.00,2,T. Rarick,1
9,visual,total,judge_total,18.30,2,S. Collins,1
